# Mount Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive/')
os.chdir('drive/MyDrive/image_forgery/')

Mounted at /content/drive/


In [ ]:
import torch
import torch.nn as nn
import numpy as np
from datetime import datetime
import os

path_authentic = f"data/recodai/train_images/authentic"
path_forged = f"data/recodai/train_images/forged"
path_mask = f"data/recodai/train_masks"
batch_size = 4
shape = 512

from torch.utils.data import DataLoader
from assets import ImgDataset, CombinedLoss

train_dataset = ImgDataset(path_authentic, path_forged, path_mask, img_shape=(shape, shape), transform=True)
test_dataset = ImgDataset(path_authentic, path_forged, path_mask,  img_shape=(shape, shape), transform=True)
cutoff = round(len(train_dataset.x) * 0.8)

train_dataset.x = train_dataset.x[:cutoff]
train_dataset.y = train_dataset.y[:cutoff]
test_dataset.x = test_dataset.x[cutoff:]
test_dataset.y = test_dataset.y[cutoff:]

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)
loss_fn = CombinedLoss()
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Code written by Hyeonwook Kim

# Models

In [ ]:
# U-NET
import torchvision.models as models
import torch.nn as nn
import torch
import torch.nn.functional as F
from collections import deque
import os
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model_name = "UNet_Ours_2"
os.makedirs(f"models/{model_name}", exist_ok=True)

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, layers=2):
        super().__init__()
        self.convs = nn.ModuleList()

        for i in range(layers):
            if i==0:
                self.convs.append(nn.Sequential(
                    nn.Conv2d(in_channels, out_channels, 3, 1, 1),
                    nn.InstanceNorm2d(out_channels),
                    nn.PReLU()))
            else:
                self.convs.append(nn.Sequential(
                    nn.Conv2d(out_channels, out_channels, 3, 1, 1),
                    nn.InstanceNorm2d(out_channels),
                    nn.PReLU()))

    def forward(self, x, y=None, upscale=False):
        B, C, H, W = x.shape

        if isinstance(y, torch.Tensor):
            x = x + y

        for i in range(0, len(self.convs)):
            if i == 0:
                x = self.convs[i](x)
            else:
                x = x + self.convs[i](x)
            deque().append(x)

        if upscale:
            x = F.interpolate(x, size=(H*2, W*2), mode='bilinear', align_corners=False)

        else:
            x = F.interpolate(x, size=(H//2, W//2), mode='bilinear', align_corners=False)

        return x

class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.down = nn.ModuleList()
        self.up = nn.ModuleList()

        self.down_layers = [3, 64, 128, 256, 512, 1024]
        self.up_layers = [1024, 512, 256, 128, 64, 64]

        for i in range(0, len(self.down_layers)-2):
            self.down.append(ConvBlock(self.down_layers[i], self.down_layers[i+1]))

        self.down.append(ConvBlock(self.down_layers[i+1], self.down_layers[i+2], layers=1))
        self.bridge = ConvBlock(self.down_layers[-1], self.down_layers[-1], layers=1)
        self.up.append(ConvBlock(self.up_layers[0], self.up_layers[1], layers=1))

        for i in range(1, len(self.up_layers)-1):
            self.up.append(ConvBlock(self.up_layers[i], self.up_layers[i+1]))

        self.out = nn.Conv2d(self.up_layers[-1], 1, 3, 1, 1)

    def forward(self, x):
        skips = deque()

        for d in self.down:
            x = d(x)
            skips.append(x)

        for u in self.up:
            x = u(x, skips.pop(), upscale=True)

        x = self.out(x)

        return x



def load_model(kwargs=None):
    return UNet().to(device)

model = load_model()
with torch.no_grad():
    x = model(torch.rand(1, 3, 512, 512).cuda())

print(x.shape)
print(sum(param.numel() for param in model.parameters() if param.requires_grad))

# Code written by Hyeonwook Kim

torch.Size([1, 1, 512, 512])
25960084


In [ ]:
# U-NET (Combined Loss)
import torchvision.models as models
import torch.nn as nn
import torch
import torch.nn.functional as F
from collections import deque
import os
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model_name = "UNet_Combined_Ours"
os.makedirs(f"models/{model_name}", exist_ok=True)

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, layers=2):
        super().__init__()
        self.convs = nn.ModuleList()

        for i in range(layers):
            if i==0:
                self.convs.append(nn.Sequential(
                    nn.Conv2d(in_channels, out_channels, 3, 1, 1),
                    nn.InstanceNorm2d(out_channels),
                    nn.PReLU()))
            else:
                self.convs.append(nn.Sequential(
                    nn.Conv2d(out_channels, out_channels, 3, 1, 1),
                    nn.InstanceNorm2d(out_channels),
                    nn.PReLU()))

    def forward(self, x, y=None, upscale=False):
        B, C, H, W = x.shape

        if isinstance(y, torch.Tensor):
            x = x + y

        for i in range(0, len(self.convs)):
            if i == 0:
                x = self.convs[i](x)
            else:
                x = x + self.convs[i](x)
            deque().append(x)

        if upscale:
            x = F.interpolate(x, size=(H*2, W*2), mode='bilinear', align_corners=False)

        else:
            x = F.interpolate(x, size=(H//2, W//2), mode='bilinear', align_corners=False)

        return x

class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.down = nn.ModuleList()
        self.up = nn.ModuleList()
        self.out = nn.ModuleList()

        self.down_layers = [3, 64, 128, 256, 512, 1024]
        self.up_layers = [1024, 512, 256, 128, 64, 32]
        self.out_layers = [512, 256, 128, 64, 32]

        for i in range(0, len(self.out_layers)):
            self.out.append(nn.Conv2d(self.out_layers[i], 1, 3, 1, 1))

        for i in range(0, len(self.down_layers)-2):
            self.down.append(ConvBlock(self.down_layers[i], self.down_layers[i+1]))

        self.down.append(ConvBlock(self.down_layers[i+1], self.down_layers[i+2], layers=1))
        self.bridge = ConvBlock(self.down_layers[-1], self.down_layers[-1], layers=1)
        self.up.append(ConvBlock(self.up_layers[0], self.up_layers[1], layers=1))

        for i in range(1, len(self.up_layers)-1):
            self.up.append(ConvBlock(self.up_layers[i], self.up_layers[i+1]))

    def forward(self, x, out_shape=(512, 512)):
        skips = deque()
        out = deque()

        for d in self.down:
            x = d(x)
            skips.append(x)

        for i in range(len(self.up)):
            x = self.up[i](x, skips.pop(), upscale=True)
            out.append(self.out[i](F.interpolate(x, size=out_shape, mode='bilinear', align_corners=False)))

        return tuple(out)

def load_model(kwargs=None):
    return UNet().to(device)

model = load_model()
with torch.no_grad():
    x = model(torch.rand(1, 3, 512, 512).cuda())

print(sum(param.numel() for param in model.parameters() if param.requires_grad))

# Code written by Hyeonwook Kim

In [ ]:
# ViT
import torchvision.models as models
import torch.nn as nn
import torch
import torch.nn.functional as F

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, layers=3):
        super().__init__()
        self.convs = nn.ModuleList()

        for i in range(layers):
            if i==0:
                self.convs.append(nn.Sequential(
                    nn.Conv2d(in_channels, out_channels, 3, 1, 1),
                    nn.BatchNorm2d(out_channels),
                    nn.PReLU()))
            else:
                self.convs.append(nn.Sequential(
                    nn.Conv2d(out_channels, out_channels, 3, 1, 1),
                    nn.BatchNorm2d(out_channels),
                    nn.PReLU()))

        self.res = nn.Sequential(
                    nn.Conv2d(in_channels, out_channels, 3, 1, 1),
                    nn.BatchNorm2d(out_channels),
                    nn.PReLU())

    def forward(self, x, y=None):

        for c in self.convs:
            x = c(x)

        if y:
            y = self.res(y)
            x += y

        B, C, H, W = x.shape
        x = F.interpolate(x, size=(H*2, W*2), mode='bilinear', align_corners=False)

        return x


class VisionTransformerEncoder(models.VisionTransformer):
    def __init__(self,
                 original_model,
                 encoder_layers=12,
                 num_heads=12,
                 decoder_layers=5,
                 image_size=512):

        super().__init__(
            image_size=image_size,
            patch_size=16,
            num_layers=encoder_layers,
            num_heads=num_heads,
            hidden_dim=original_model.hidden_dim, # 768
            mlp_dim=original_model.mlp_dim,
            dropout=original_model.dropout,
            attention_dropout=original_model.attention_dropout,
            num_classes=original_model.num_classes)

        self.conv = nn.ModuleList([ConvBlock(original_model.hidden_dim // 2**i, original_model.hidden_dim // 2**(i+1)) for i in range(decoder_layers)])
        self.out = nn.Conv2d(original_model.hidden_dim // 2**decoder_layers, 1, 3, 1, 1)
        self.image_size = image_size

    def forward(self, x: torch.Tensor):
        # Reshape and permute the input tensor
        x = self._process_input(x)
        n = x.shape[0]

        # Expand the class token to the full batch
        batch_class_token = self.class_token.expand(n, -1, -1)
        x = torch.cat([batch_class_token, x], dim=1)

        x = self.encoder(x)

        # Classifier "token" as used by standard language architectures
        x = x[:, 1:]
        x = x.permute(0, 2, 1)

        B, C, D = x.shape
        S = int(D ** 0.5)
        x = x.reshape(B, C, S, S)

        for c in self.conv:
            x = c(x)

        x = self.out(x)
        x = F.interpolate(x, size=(self.image_size, self.image_size), mode='bilinear', align_corners=False)

        return x


def load_model(kwargs=None):
    return VisionTransformerEncoder(models.vit_b_16()).to(device)

model = load_model()
model_name = "ViTSegmentModel_2"
print(sum(param.numel() for param in model.parameters() if param.requires_grad))

# Code written by Hyeonwook Kim

# Other Models

In [ ]:
# CAT-NET
import os
#! git clone https://github.com/aL3x-O-o-Hung/CAT-Net
model_name = "CAT-Net"
os.chdir(model_name)

from CAT_Net import CrossSliceAttentionUNet

def load_model(kwargs=None, shape=shape, batch_size=batch_size):
    input_channels=3
    num_classes=1
    num_layers=6
    heads=1
    num_attention_blocks=1
    base_num=64
    pool_kernel_size=4
    input_size=shape
    batch_size=batch_size
    pool_method="avgpool"
    is_pe_learnable=True
    return CrossSliceAttentionUNet(input_channels,
                                  num_classes,
                                  num_layers,
                                  heads,
                                  num_attention_blocks,
                                  base_num,
                                  (pool_kernel_size,pool_kernel_size),
                                  (input_size,input_size),
                                  batch_size,
                                  pool_method,
                                  is_pe_learnable).to(device)

model = load_model()
os.chdir('..')

# Code written by Hyeonwook Kim

In [ ]:
# MicroSegNet
#! git clone https://github.com/mirthAI/MicroSegNet
! pip install ml-collections tensorboardX medpy SimpleITK
import os
model_name = "MicroSegNet"
def load_model():
    os.chdir("MicroSegNet/TransUNet")
    from networks.vit_seg_modeling import VisionTransformer as ViT_seg
    from networks.vit_seg_modeling import CONFIGS as CONFIGS_ViT_seg
    config_vit = CONFIGS_ViT_seg["R50-ViT-B_16"]
    config_vit.n_classes = 1
    config_vit.n_skip = 3
    img_size = 224
    config_vit.patches.grid = (int(img_size / 16), int(img_size / 16))
    model = ViT_seg(config_vit, img_size=224, num_classes=config_vit.n_classes).cuda()
    os.chdir("..")
    os.chdir("..")
    return model


model = load_model()

# Code written by Hyeonwook Kim

# Optimizer

In [ ]:
import torch.optim as optim
optimizer = optim.AdamW(
    model.parameters(),
    lr=5e-4,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=1e-4,  # L2 regularization
)

# Code written by Hyeonwook Kim

# Train Loop

In [ ]:
epochs = 300
min_cum_loss = 10
os.makedirs(f"models/{model_name}", exist_ok=True)

for e in range(epochs):
    model.train()
    cum_loss = []
    for train_x, train_y in train_dataloader:
        if train_x.shape[0]==batch_size:
            train_x = train_x.to(device)
            train_y = train_y.to(device)
            optimizer.zero_grad()
            pred_y = model(train_x)
            loss = loss_fn(pred_y, train_y)
            loss.backward()
            optimizer.step()
            cum_loss.append(loss.item())

    print(e+1, round(np.mean(cum_loss).item(), 4))
    with torch.no_grad():
        model.eval()
        cum_loss = []
        cum_dce = []
        cum_iou = []

        for test_x, test_y in test_dataloader:
            if test_x.shape[0]==batch_size:
                test_x = test_x.to(device)
                test_y = test_y.to(device)
                pred_y = model(test_x)

                cum_loss.append(loss_fn(pred_y, test_y).item())
                cum_dce.append(loss_fn.dice_loss(pred_y, test_y).item())
                cum_iou.append(loss_fn.iou_loss(pred_y, test_y).item())

        cur_loss = np.mean(cum_loss).item()

        print("Loss:",round(cur_loss, 4))
        print("DICE:",round(1 - np.mean(cum_dce).item(), 4))
        print("IoU:",round(1 - np.mean(cum_iou).item(), 4))
        print(datetime.now())

        if min_cum_loss > cur_loss:
            min_cum_loss = cur_loss
            torch.save(model.state_dict(), f"models/{model_name}/{round(cur_loss, 4)}_{e+1:04d}.pth")

        print()

# Code written by Hyeonwook Kim

1 2.0015
Loss: 1.8158
DICE: 0.1879
IoU: 0.1127
2025-11-22 06:16:14.478704

2 1.9373
Loss: 1.769
DICE: 0.2141
IoU: 0.1307
2025-11-22 06:23:47.608535

3 1.925
Loss: 1.7486
DICE: 0.226
IoU: 0.1408
2025-11-22 06:31:20.571652

4 1.9116
Loss: 1.7135
DICE: 0.2457
IoU: 0.1559
2025-11-22 06:38:53.883501

5 1.8993
Loss: 1.6343
DICE: 0.2888
IoU: 0.1936
2025-11-22 06:46:26.506740

6 1.8835
Loss: 1.7443
DICE: 0.227
IoU: 0.1491
2025-11-22 06:53:59.657204

7 1.8702
Loss: 1.6227
DICE: 0.2913
IoU: 0.1956
2025-11-22 07:01:32.000100

8 1.8551
Loss: 1.5644
DICE: 0.3262
IoU: 0.225
2025-11-22 07:09:04.518185

9 1.8484
Loss: 1.5819
DICE: 0.3144
IoU: 0.2157
2025-11-22 07:16:38.664258

10 1.8403
Loss: 1.5815
DICE: 0.3173
IoU: 0.217
2025-11-22 07:24:11.265845

11 1.8302
Loss: 1.5622
DICE: 0.3245
IoU: 0.2234
2025-11-22 07:31:44.138598

12 1.8302
Loss: 1.5409
DICE: 0.3362
IoU: 0.2313
2025-11-22 07:39:18.003738

13 1.8205
Loss: 1.5891
DICE: 0.31
IoU: 0.2118
2025-11-22 07:46:52.022054

14 1.8149
Loss: 1.5088
DICE: 

In [ ]:
epochs = 300
min_cum_loss = 10

for e in range(epochs):
    model.train()
    cum_loss = []
    for train_x, train_y in train_dataloader:
        if train_x.shape[0]==batch_size:
            train_x = train_x.to(device)
            train_y = train_y.to(device)
            optimizer.zero_grad()
            pred_y = model(train_x)
            loss = loss_fn(pred_y[0], train_y)
            loss.backward()
            optimizer.step()
            cum_loss.append(loss.item())

    print(e+1, round(np.mean(cum_loss).item(), 4))
    with torch.no_grad():
        model.eval()
        cum_loss = []
        cum_dce = []
        cum_iou = []

        for test_x, test_y in test_dataloader:
            if test_x.shape[0]==batch_size:
                test_x = test_x.to(device)
                test_y = test_y.to(device)
                pred_y = model(test_x)

                cum_loss.append(loss_fn(pred_y[0], test_y).item())
                cum_dce.append(loss_fn.dice_loss(pred_y[0], test_y).item())
                cum_iou.append(loss_fn.iou_loss(pred_y[0], test_y).item())

        cur_loss = np.mean(cum_loss).item()

        print("Loss:",round(cur_loss, 4))
        print("DICE:",round(1 - np.mean(cum_dce).item(), 4))
        print("IoU:",round(1 - np.mean(cum_iou).item(), 4))
        print(datetime.now())

        if min_cum_loss > cur_loss:
            min_cum_loss = cur_loss
            torch.save(model.state_dict(), f"models/{model_name}/{round(cur_loss, 4)}_{e+1:04d}.pth")

        print()

# Code written by Hyeonwook Kim

# View Image

In [ ]:
i=133
with torch.no_grad():
    model.eval()
    x, y = test_dataset[i]
    x, y = torch.unsqueeze(x, 0).to(device), torch.unsqueeze(y, 0).to(device)
    pred_y = model(x)

import torch
import numpy as np
import matplotlib.pyplot as plt

C, H, W = 3, 64, 64
tensor1 = x[0]
tensor2 = y[0]
tensor3 = nn.Sigmoid()(pred_y)[0]

tensors = [tensor1, tensor2, tensor3]
titles = ["Image 1", "Image 2", "Image 3"]
numpy_arrays = []

for tensor in tensors:
    np_array = tensor.detach().cpu().numpy()
    np_image = np.transpose(np_array, (1, 2, 0))

    numpy_arrays.append(np_image)

fig, axes = plt.subplots(1, 3, figsize=(12, 4)) #

for i, img in enumerate(numpy_arrays):
    axes[i].imshow(img)
    axes[i].set_title(titles[i])
    axes[i].axis('off')

plt.tight_layout()
plt.show()

# Code written by Hyeonwook Kim